In [3]:
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
import pandas as pd
import numpy as np
import pickle
from natsort import natsorted

import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
import seaborn as sns
sns.set(style='ticks', context='notebook', font_scale=1.2)


def rm_termini(m):
    
    # rm all Br
    m = protect_CBr(m)
    while m.HasSubstructMatch(Chem.MolFromSmarts('cBr')):
        smarts = "[*:1]Br>>[*:1]"
        rxn = AllChem.ReactionFromSmarts(smarts)
        ps = rxn.RunReactants((m,))
        products = rm_duplicate_mols([m[0] for m in ps])
        m = products[0]
    m = deprotect_CBr(m)
    
    # rm all BOO
    while m.HasSubstructMatch(Chem.MolFromSmarts('[B](-O)(-O)')):
        smarts = "[*:1]([B](-O)(-O))>>[*:1]"
        rxn = AllChem.ReactionFromSmarts(smarts)
        ps = rxn.RunReactants((m,))
        products = rm_duplicate_mols([m[0] for m in ps])
        m = products[0]
        
    return m

def rm_duplicate_mols(mols):
    smiles = list(set([Chem.MolToSmiles(m, canonical=True) for m in mols]))
    mols = [Chem.MolFromSmiles(s) for s in smiles]
    return mols


def protect_CBr(m):
    while m.HasSubstructMatch(Chem.MolFromSmarts('cCBr')):
        smarts = "[*:1]CBr>>[*:1]C[At]"
        rxn = AllChem.ReactionFromSmarts(smarts)
        ps = rxn.RunReactants((m,))
        products = rm_duplicate_mols([m[0] for m in ps])
        m = products[0]
    return m


def deprotect_CBr(m):
    while m.HasSubstructMatch(Chem.MolFromSmarts('C[At]')):
        smarts = "[*:1]C[At]>>[*:1]CBr"
        rxn = AllChem.ReactionFromSmarts(smarts)
        ps = rxn.RunReactants((m,))
        products = rm_duplicate_mols([m[0] for m in ps])
        m = products[0]
    return m


def rm_termini(m):
    
    # rm all Br
    m = protect_CBr(m)
    while m.HasSubstructMatch(Chem.MolFromSmarts('cBr')):
        smarts = "[*:1]Br>>[*:1]"
        rxn = AllChem.ReactionFromSmarts(smarts)
        ps = rxn.RunReactants((m,))
        products = rm_duplicate_mols([m[0] for m in ps])
        m = products[0]
    m = deprotect_CBr(m)
    
    # rm all BOO
    while m.HasSubstructMatch(Chem.MolFromSmarts('[B](-O)(-O)')):
        smarts = "[*:1]([B](-O)(-O))>>[*:1]"
        rxn = AllChem.ReactionFromSmarts(smarts)
        ps = rxn.RunReactants((m,))
        products = rm_duplicate_mols([m[0] for m in ps])
        m = products[0]
        
    return m



# Load the Excel file into a pandas DataFrame
df = pd.read_excel("/Users/elahehkazemi/Desktop/smiles/dataset.xlsx")

# Save the DataFrame to a CSV file
df.to_csv("dataset.csv", index=False)  # Set index=False to exclude row numbers in the CSV

def make_master_chemprop_input(smiA, smiB):
    mA = Chem.MolFromSmiles(smiA)
    mB = Chem.MolFromSmiles(smiB)
    mA = rm_termini(mA)
    mB = rm_termini(mB)
    smiA = Chem.MolToSmiles(mA, canonical=True)
    smiB = Chem.MolToSmiles(mB, canonical=True)
    smiles = f'{smiA}.{smiB}'
    return smiles


In [4]:
# ==============================
# Master + Poly Chemprop inputs
# ==============================

df.loc[:, 'master_chemprop_input'] = [make_master_chemprop_input(sA, sB) for sA, sB in zip(df.loc[:, 'monoA'], df.loc[:, 'monoB'])]
df

,Copolymer Name,poly_type,monoA,monoB,%A,fracA,%B,fracB,Density(gcm-³),Rg,...,compressibility(Pa-1),isentropic_compressibility(Pa-1),bulk_modulus(Pa),isentropic_bulk_modulus(Pa),volume_expansion(k-1),linear_expansion(k-1),r2,static_dielectric_const,nematic_order_parameter,master_chemprop_input
0,70ethene_30butenemethylpentene_acopoly,alternating,CCCCCCNC(=O)CCCCC(=O)N,CCCCCCNC(=O)c1ccc(cc1)C(=O)N,70,0.70,30,0.30,0.829473,14.216247,...,4.688193e-10,4.306815e-10,2.142295e+09,2.327113e+09,0.000719,0.000240,10.594306,1.013504,0.040607,CCCCCCNC(=O)CCCCC(N)=O.CCCCCCNC(=O)c1ccc(C(N)=...
1,53PVC_47vinylacetate_rcopoly,random,CC(Cl),C(OC(=O)C)C,53,0.53,47,0.47,1.187662,19.300580,...,4.280789e-10,3.967904e-10,2.336871e+09,2.521173e+09,0.000595,0.000198,19.429984,1.481721,0.026518,CCCl.CCOC(C)=O
2,50bisphenolbiphenol_50dichlorodiphenylsulfone_...,alternating,Oc1ccc(cc1)S(=O)(=O)c1ccc(cc1),Oc1ccc(cc1)S(=O)(=O)c1ccc(cc1)Oc1ccc(cc1)c1ccc...,50,0.50,50,0.50,1.260426,28.639150,...,3.105790e-10,2.994245e-10,3.221087e+09,3.340431e+09,0.000350,0.000117,44.894528,1.293329,0.028751,O=S(=O)(c1ccccc1)c1ccc(O)cc1.O=S(=O)(c1ccc(O)c...
3,50PE_50PP_Bcopoly,block,C,CC(C),50,0.50,50,0.50,0.817728,11.293852,...,4.824279e-10,4.727361e-10,2.073150e+09,2.115656e+09,0.000351,0.000117,10.882164,1.051868,0.052924,C.CCC
4,90bistrimethylsilylphenylenediaminebistrimethy...,alternating,Nc1ccc(cc1)NC(=O)c1cccc(c1)C(=O)c1ccc(cc1)C(=O),Oc1cccc(c1)NC(=O)c1ccc(cc1)C(=O)c1cccc(c1)C(=O...,90,0.90,10,0.10,1.229430,29.008510,...,2.485962e-10,2.452457e-10,4.023762e+09,4.078660e+09,0.000201,0.000067,34.566867,1.615726,0.037647,Nc1ccc(NC(=O)c2cccc(C(=O)c3ccc(C=O)cc3)c2)cc1....
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,85PE_15BUT_Bcopoly_second,block,C,CC(CC),85,0.85,15,0.15,0.964636,18.475673,...,3.560319e-10,3.406908e-10,2.809523e+09,2.936270e+09,0.000454,0.000151,17.882920,1.979061,0.031884,C.CCCC
136,90ethene_10butenemethylpentene_acopoly,alternating,CCCC(CC),CC(C(CC)C),90,0.90,10,0.10,0.829944,12.987932,...,4.493973e-10,4.408313e-10,2.227323e+09,2.270594e+09,0.000332,0.000111,8.767701,1.013989,0.034389,CCCCCC.CCC(C)CC
137,86dimetyl_14diheptyl_rcopoly_second,random,COC(=O)CC(C(=O)OC)(C),CCCCCCCOC(=O)CC(C(=O)OCCCCCCC)(C),86,0.86,14,0.14,1.093024,21.957822,...,3.299941e-10,3.242284e-10,3.030482e+09,3.084336e+09,0.000270,0.000090,22.776606,1.423900,0.048407,COC(=O)CC(C)C(=O)OC.CCCCCCCOC(=O)CC(C)C(=O)OCC...
138,75biphenyltetracarboxylicdianhydride_25phenyle...,alternating,c1ccc(cc1)Oc1ccc(cc1)N1C(=O)c2c(C1=O)cc(cc2)c1...,c1ccc(cc1)N1C(=O)c2c(C1=O)cc(cc2)c1ccc2c(c1)C(...,75,0.75,25,0.25,0.982999,31.268319,...,7.845612e-10,6.895642e-10,1.274763e+09,1.450245e+09,0.001210,0.000403,0.000000,80.869850,0.248516,O=C1NC(=O)c2c1cccc2-c1ccc2c(c1)C(=O)N(c1ccc(Oc...


In [12]:
def make_poly_chemprop_input(smiA, smiB, poly_type, fracA=0.5, selfedges=True):
    
    
   
    mA = Chem.MolFromSmiles(smiA)
    mB = Chem.MolFromSmiles(smiB)
    mA = rm_termini(mA)
    mB = rm_termini(mB)
    smiA = Chem.MolToSmiles(mA, canonical=True)
    smiB = Chem.MolToSmiles(mB, canonical=True)
    smiles = f'{smiA}.{smiB}'
    
    # fractions of monomers
    fracB = 1.0 - fracA
    fracs = f'|{fracA}|{fracB}|'
    
    # specify extra edges
    if poly_type == 'alternating':
        # 1-3
        # 1-4
        # 2-3
        # 2-4
        edges = '<1-3:0.5:0.5<1-4:0.5:0.5<2-3:0.5:0.5<2-4:0.5:0.5'
    elif poly_type == 'block':
        # between: 1-3, 1-4, 2-3, 2-4 (weight 1/7)
        # within: 1-2, 3-4
        # self: 1-1, 2-2, 3-3, 4-4
        if selfedges is True:
            edges = [(1, 2, 3/8, 3/8),  # within A
                     (1, 1, 3/8, 3/8),
                     (2, 2, 3/8, 3/8),
                     (3, 4, 3/8, 3/8),  # within B
                     (3, 3, 3/8, 3/8),
                     (4, 4, 1/8, 1/8),
                     (1, 3, 1/8, 1/8),  # between A and B
                     (1, 4, 1/8, 1/8),
                     (2, 3, 1/8, 1/8),
                     (2, 4, 1/8, 1/8)]
        else:
            edges = [(1, 2, 6/8, 6/8),  # within A
                     (3, 4, 6/8, 6/8),  # within B
                     (1, 3, 1/8, 1/8),  # between A and B
                     (1, 4, 1/8, 1/8),
                     (2, 3, 1/8, 1/8),
                     (2, 4, 1/8, 1/8)]
        edges = "".join([f"<{e[0]}-{e[1]}:{e[2]}:{e[3]}" for e in edges])
    elif poly_type == 'random':
        # between: 1-3, 1-4, 2-3, 2-4
        # within: 1-2, 3-4
        # self: 1-1, 2-2, 3-3, 4-4
        if selfedges is True:
            edges = '<1-3:0.25:0.25<1-4:0.25:0.25<2-3:0.25:0.25<2-4:0.25:0.25<1-2:0.25:0.25<3-4:0.25:0.25<1-1:0.25:0.25<2-2:0.25:0.25<3-3:0.25:0.25<4-4:0.25:0.25'
        else:
            edges = '<1-3:0.25:0.25<1-4:0.25:0.25<2-3:0.25:0.25<2-4:0.25:0.25<1-2:0.5:0.5<3-4:0.5:0.5'
    
    return f"{smiles}{fracs}{edges}"


In [13]:
df.loc[:, 'poly_chemprop_input'] = [make_poly_chemprop_input(sA, sB, t, fA, selfedges=True) for 
                                    sA, sB, t, fA in 
                                    zip(df.loc[:, 'monoA'], df.loc[:, 'monoB'], df.loc[:, 'poly_type'], df.loc[:, 'fracA'])]
df

,Copolymer Name,poly_type,monoA,monoB,%A,fracA,%B,fracB,Density(gcm-³),Rg,...,isentropic_compressibility(Pa-1),bulk_modulus(Pa),isentropic_bulk_modulus(Pa),volume_expansion(k-1),linear_expansion(k-1),r2,static_dielectric_const,nematic_order_parameter,master_chemprop_input,poly_chemprop_input
0,70ethene_30butenemethylpentene_acopoly,alternating,CCCCCCNC(=O)CCCCC(=O)N,CCCCCCNC(=O)c1ccc(cc1)C(=O)N,70,0.70,30,0.30,0.829473,14.216247,...,4.306815e-10,2.142295e+09,2.327113e+09,0.000719,0.000240,10.594306,1.013504,0.040607,CCCCCCNC(=O)CCCCC(N)=O.CCCCCCNC(=O)c1ccc(C(N)=...,CCCCCCNC(=O)CCCCC(N)=O.CCCCCCNC(=O)c1ccc(C(N)=...
1,53PVC_47vinylacetate_rcopoly,random,CC(Cl),C(OC(=O)C)C,53,0.53,47,0.47,1.187662,19.300580,...,3.967904e-10,2.336871e+09,2.521173e+09,0.000595,0.000198,19.429984,1.481721,0.026518,CCCl.CCOC(C)=O,CCCl.CCOC(C)=O|0.53|0.47|<1-3:0.25:0.25<1-4:0....
2,50bisphenolbiphenol_50dichlorodiphenylsulfone_...,alternating,Oc1ccc(cc1)S(=O)(=O)c1ccc(cc1),Oc1ccc(cc1)S(=O)(=O)c1ccc(cc1)Oc1ccc(cc1)c1ccc...,50,0.50,50,0.50,1.260426,28.639150,...,2.994245e-10,3.221087e+09,3.340431e+09,0.000350,0.000117,44.894528,1.293329,0.028751,O=S(=O)(c1ccccc1)c1ccc(O)cc1.O=S(=O)(c1ccc(O)c...,O=S(=O)(c1ccccc1)c1ccc(O)cc1.O=S(=O)(c1ccc(O)c...
3,50PE_50PP_Bcopoly,block,C,CC(C),50,0.50,50,0.50,0.817728,11.293852,...,4.727361e-10,2.073150e+09,2.115656e+09,0.000351,0.000117,10.882164,1.051868,0.052924,C.CCC,C.CCC|0.5|0.5|<1-2:0.375:0.375<1-1:0.375:0.375...
4,90bistrimethylsilylphenylenediaminebistrimethy...,alternating,Nc1ccc(cc1)NC(=O)c1cccc(c1)C(=O)c1ccc(cc1)C(=O),Oc1cccc(c1)NC(=O)c1ccc(cc1)C(=O)c1cccc(c1)C(=O...,90,0.90,10,0.10,1.229430,29.008510,...,2.452457e-10,4.023762e+09,4.078660e+09,0.000201,0.000067,34.566867,1.615726,0.037647,Nc1ccc(NC(=O)c2cccc(C(=O)c3ccc(C=O)cc3)c2)cc1....,Nc1ccc(NC(=O)c2cccc(C(=O)c3ccc(C=O)cc3)c2)cc1....
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,85PE_15BUT_Bcopoly_second,block,C,CC(CC),85,0.85,15,0.15,0.964636,18.475673,...,3.406908e-10,2.809523e+09,2.936270e+09,0.000454,0.000151,17.882920,1.979061,0.031884,C.CCCC,C.CCCC|0.85|0.15000000000000002|<1-2:0.375:0.3...
136,90ethene_10butenemethylpentene_acopoly,alternating,CCCC(CC),CC(C(CC)C),90,0.90,10,0.10,0.829944,12.987932,...,4.408313e-10,2.227323e+09,2.270594e+09,0.000332,0.000111,8.767701,1.013989,0.034389,CCCCCC.CCC(C)CC,CCCCCC.CCC(C)CC|0.9|0.09999999999999998|<1-3:0...
137,86dimetyl_14diheptyl_rcopoly_second,random,COC(=O)CC(C(=O)OC)(C),CCCCCCCOC(=O)CC(C(=O)OCCCCCCC)(C),86,0.86,14,0.14,1.093024,21.957822,...,3.242284e-10,3.030482e+09,3.084336e+09,0.000270,0.000090,22.776606,1.423900,0.048407,COC(=O)CC(C)C(=O)OC.CCCCCCCOC(=O)CC(C)C(=O)OCC...,COC(=O)CC(C)C(=O)OC.CCCCCCCOC(=O)CC(C)C(=O)OCC...
138,75biphenyltetracarboxylicdianhydride_25phenyle...,alternating,c1ccc(cc1)Oc1ccc(cc1)N1C(=O)c2c(C1=O)cc(cc2)c1...,c1ccc(cc1)N1C(=O)c2c(C1=O)cc(cc2)c1ccc2c(c1)C(...,75,0.75,25,0.25,0.982999,31.268319,...,6.895642e-10,1.274763e+09,1.450245e+09,0.001210,0.000403,0.000000,80.869850,0.248516,O=C1NC(=O)c2c1cccc2-c1ccc2c(c1)C(=O)N(c1ccc(Oc...,O=C1NC(=O)c2c1cccc2-c1ccc2c(c1)C(=O)N(c1ccc(Oc...


In [14]:
df.to_excel("output.xlsx", index=False)  # Set index=False to exclude row numbers in the Excel

print("DataFrame has been saved as output.xlsx")

DataFrame has been saved as output.xlsx
